This notebook takes the output of the ortho creation step & precomputes wald

In [2]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

2025-11-07 16:30:23.152765: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-07 16:30:23.355601: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Set up the cluster

In [3]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=5,#cores per slurm job
        memory="64G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=0:20:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=2)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

Let's just do a super simple bounded concurrency approach

In [4]:
from pathlib import Path

In [5]:
# in the real version, de_novo_sim will take a pair, path/name on init and never save it.
# relative paths to individual components can be used, saved, assumed. 
DATA_ROOT=Path("/home/mcn26/project_pi_skr2/shared/tabula_data")
path=DATA_ROOT/"simulated"
name="shendure_calibrated_sim_with_orthos_20251008"

In [6]:
from dask.distributed import Semaphore, as_completed, get_client

In [7]:
ortho_root=path/name/"orthos"
scmpradat_root=path/name/"scMPRA"
output_root=path/name/"orthos_with_precomputed_wald_erin_test_graph_mode"
output_root.mkdir(exist_ok=True)

input_ortho_names=[path.name for path in ortho_root.iterdir()]

Semaphore(max_leases=2, name="wald-precompute")

def precompute_one_wald(input_root, scmpradat_root, name, output_root):
    sem = Semaphore(name="wald-precompute")
    with sem:
        client=get_client()
        dat=scm.scMPRA_data.from_parquet(scmpradat_root/Path(name).with_suffix(".scmpra"))
        dat.ortho_filter()
        ortho_oi=scm.ortho.load(client=client,
                                path=input_root,
                                name=name)
        ortho_oi.training_data=dat
        ortho_oi.precompute_wald(client)
        return ortho_oi

test_particle=precompute_one_wald(input_root=ortho_root,
    scmpradat_root=scmpradat_root,
    name=input_ortho_names[0],
    output_root=output_root)
#futures = [client.submit(precompute_one_wald,) for name_oi in input_ortho_names]

scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [8]:
import pandas as pd

In [9]:
names=[]
errors=[]
for name in test_particle.wald_precomp.by_cell_type:
    names.append(name)
    errors.append(test_particle.wald_precomp.by_cell_type[name].result().debug_msg)
pd.DataFrame({"name":names,"debug":errors}).to_csv("by_cell_types.tsv",sep="\t")

In [10]:
names=[]
errors=[]
for name in test_particle.wald_precomp.by_cre:
    names.append(name)
    errors.append(test_particle.wald_precomp.by_cre[name].result().debug_msg)
pd.DataFrame({"name":names,"debug":errors}).to_csv("by_cre.tsv",sep="\t")

In [ ]:
client.close()
cluster.close()

In [ ]:
!rm ./worker*

In [ ]:
#!rm -r $output_root